# Used Car Condition Classification

**Tabular Classification Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `condition`

## 2 · Project Overview

We classify **used car condition** into 3 levels (Poor, Fair, Good) based on age, mileage, ownership history, service records, accident history, fuel type, and transmission.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular classification dataset.
2. Encode categorical features for tree-based models.
3. Build a baseline Logistic Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given a used car's age, mileage, number of owners, service history, accident record, fuel type, and transmission, classify its overall condition.

## 5 · Why This Project Matters

- **Used car valuation** is a multi-billion dollar industry.
- Condition assessment drives pricing and buyer confidence.
- Automated classification scales to thousands of listings.
- Understanding condition drivers helps sellers present cars better.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 7,000 |
| **Features** | age_years, mileage_k, num_owners, service_records, accident_history, fuel_type, transmission |
| **Target** | condition (3 classes: Poor, Fair, Good) |
| **Class balance** | ~40/35/25 |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "condition"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

Target: condition
Save dir: E:\Github\Machine-Learning-Projects\Classification\Used Car Condition Classification


## 11 · Dataset Download or Loading

Synthetic dataset: 7,000 used cars with vehicle attributes and 3-level condition rating.

In [4]:
np.random.seed(SEED)
n = 7000
age_years = np.random.randint(1, 20, n)
mileage_k = np.round(age_years * np.random.uniform(8, 18, n) + np.random.normal(0, 10, n), 1).clip(1, 350)
num_owners = np.random.poisson(1.5, n).clip(1, 8)
service_records = np.random.poisson(age_years * 1.2, n).clip(0, 40)
accident_history = np.random.binomial(1, 0.2, n)
fuel_type = np.random.choice(["Petrol", "Diesel", "Hybrid", "Electric"], n, p=[0.4, 0.3, 0.15, 0.15])
transmission = np.random.choice(["Manual", "Automatic"], n, p=[0.4, 0.6])

score = (-0.08 * age_years - 0.003 * mileage_k - 0.15 * num_owners
         + 0.03 * service_records - 0.5 * accident_history + np.random.normal(0, 0.7, n))
condition = np.where(score > np.percentile(score, 60), "Good",
            np.where(score > np.percentile(score, 25), "Fair", "Poor"))

for level in ["Good", "Fair", "Poor"]:
    if (condition == level).sum() < 20:
        idxs = np.random.choice(n, 20, replace=False)
        condition[idxs[:max(0, 20-(condition==level).sum())]] = level

df = pd.DataFrame({"age_years": age_years, "mileage_k": mileage_k,
                    "num_owners": num_owners, "service_records": service_records,
                    "accident_history": accident_history, "fuel_type": fuel_type,
                    "transmission": transmission, "condition": condition})
print(f"Dataset shape: {df.shape}")
print(f"Class balance:\n{df['condition'].value_counts(normalize=True).round(3)}")
df.head()

Dataset shape: (7000, 8)
Class balance:
condition
Good    0.40
Fair    0.35
Poor    0.25
Name: proportion, dtype: float64


,age_years,mileage_k,num_owners,service_records,accident_history,fuel_type,transmission,condition
0,7,119.4,1,3,0,Petrol,Manual,Good
1,15,150.2,1,19,0,Petrol,Manual,Fair
2,11,171.0,2,12,0,Diesel,Manual,Poor
3,8,142.0,2,9,0,Diesel,Automatic,Poor
4,7,81.5,2,10,0,Petrol,Automatic,Good


## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed. Value counts:")
print(df[TARGET].value_counts())

DATA VALIDATION

Shape: (7000, 8)

Missing values:
age_years           0
mileage_k           0
num_owners          0
service_records     0
accident_history    0
fuel_type           0
transmission        0
condition           0
dtype: int64

Duplicate rows: 11

Dtypes:
age_years             int32
mileage_k           float64
num_owners            int32
service_records       int32
accident_history      int32
fuel_type            object
transmission         object
condition            object
dtype: object

Target 'condition' confirmed. Value counts:
condition
Good    2800
Fair    2450
Poor    1750
Name: count, dtype: int64


## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [6]:
num_cols = ["age_years", "mileage_k", "num_owners", "service_records"]
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for i, col in enumerate(num_cols):
    ax = axes[i // 2][i % 2]
    df.boxplot(column=col, by=TARGET, ax=ax)
    ax.set_title(col)
plt.suptitle("Feature Distributions by Car Condition", fontsize=14)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df.groupby("fuel_type")[TARGET].value_counts(normalize=True).unstack().plot(kind="bar", ax=axes[0], edgecolor="black")
axes[0].set_title("Condition by Fuel Type")
axes[0].legend(fontsize=8)
df.groupby("accident_history")[TARGET].value_counts(normalize=True).unstack().plot(kind="bar", ax=axes[1], edgecolor="black")
axes[1].set_title("Condition by Accident History")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `condition`.

In [7]:
fig, ax = plt.subplots(figsize=(6, 4))
order = ["Poor", "Fair", "Good"]
df[TARGET].value_counts().reindex(order).plot(kind="bar", ax=ax,
    color=["salmon", "orange", "steelblue"], edgecolor="black")
ax.set_title("Car Condition Distribution")
plt.tight_layout()
plt.show()
print(f"Condition counts:\n{df[TARGET].value_counts()}")

Condition counts:
condition
Good    2800
Fair    2450
Poor    1750
Name: count, dtype: int64


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 train/test split preserving class balance.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

# Encode target if needed
if y.dtype == object:
    le = LabelEncoder()
    y = pd.Series(le.fit_transform(y), name=TARGET)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train class distribution:\n{y_train.value_counts(normalize=True).round(3)}")

Train: (5600, 7), Test: (1400, 7)
Train class distribution:
condition
1    0.40
0    0.35
2    0.25
Name: proportion, dtype: float64


## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: `fuel_type`, `transmission` encoded with OrdinalEncoder. Target encoded with LabelEncoder.
- **Scaling**: Not needed for tree models.
- **Class balance**: ~40/35/25 (Good/Fair/Poor).

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [9]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["mileage_per_year"] = X_train["mileage_k"] / (X_train["age_years"] + 1)
X_test["mileage_per_year"] = X_test["mileage_k"] / (X_test["age_years"] + 1)

X_train["service_per_year"] = X_train["service_records"] / (X_train["age_years"] + 1)
X_test["service_per_year"] = X_test["service_records"] / (X_test["age_years"] + 1)

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (9): ['age_years', 'mileage_k', 'num_owners', 'service_records', 'accident_history', 'fuel_type', 'transmission', 'mileage_per_year', 'service_per_year']


## 18 · Baseline Model

Logistic Regression as our baseline classifier.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

acc_bl = accuracy_score(y_test, y_pred_bl)
f1_bl = f1_score(y_test, y_pred_bl, average="weighted")

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {acc_bl:.4f}")
print(f"F1       : {f1_bl:.4f}")
print("\n" + classification_report(y_test, y_pred_bl))

BASELINE: Logistic Regression
Accuracy : 0.5821
F1       : 0.5770

              precision    recall  f1-score   support

           0       0.48      0.44      0.46       490
           1       0.64      0.74      0.69       560
           2       0.60      0.53      0.56       350

    accuracy                           0.58      1400
   macro avg       0.58      0.57      0.57      1400
weighted avg       0.58      0.58      0.58      1400



## 19 · LazyPredict Benchmark

Fit dozens of classifiers in one call for a quick ranking.

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                               Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                          
LogisticRegression             0.581429           0.568622  0.743057  0.575368   0.574435  0.581429    0.039466
CalibratedClassifierCV         0.577143           0.565017  0.738876  0.564165   0.564422  0.577143    0.098791
LinearDiscriminantAnalysis     0.574286           0.562398  0.741893  0.569919   0.569011  0.574286    0.020661
LinearSVC                      0.568571           0.559014       NaN  0.545612   0.553052  0.568571    0.025864
RidgeClassifierCV              0.567857           0.557789       NaN  0.544928   0.554085  0.567857    0.019785
RidgeClassifier                0.567857           0.557432       NaN  0.544764   0.554482  0.567857    0.017435
AdaBoostClassifier             0.568571           0.556905  0.717929  

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="classification", time_budget=60,
                 metric="macro_f1",
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1: {f1_score(y_test, y_pred_flaml, average='weighted'):.4f}")

FLAML best model: lgbm
Accuracy: 0.5714
F1: 0.5693


## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [13]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostClassifier
    t0 = time.perf_counter()
    try:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test).ravel()
    print(f"CatBoost F1: {f1_score(y_test, results['CatBoost'], average='weighted'):.4f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM F1: {f1_score(y_test, results['LightGBM'], average='weighted'):.4f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBClassifier
    t0 = time.perf_counter()
    try:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost F1: {f1_score(y_test, results['XGBoost'], average='weighted'):.4f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost F1: 0.5658  (1.8s)


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[61]	valid_0's multi_logloss: 0.909509
LightGBM F1: 0.5684  (1.9s)


XGBoost failed: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'


## 22 · Top 3 Model Selection

Rank all models by F1 and select the top 3.

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average="weighted"), 4),
        "Precision": round(precision_score(y_test, yp, average="weighted", zero_division=0), 4),
        "Recall": round(recall_score(y_test, yp, average="weighted", zero_division=0), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
Logistic Regression    0.5821  0.5770     0.5762  0.5821
FLAML                  0.5714  0.5693     0.5711  0.5714
LightGBM               0.5707  0.5684     0.5690  0.5707
CatBoost               0.5679  0.5658     0.5684  0.5679

Top 3 models: ['Logistic Regression', 'FLAML', 'LightGBM']


## 23 · Final Training and Evaluation of Top 3

Detailed metrics and confusion matrices.

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap="Blues")
    axes[i].set_title(f"{name}\nF1={f1_score(y_test, yp, average='weighted'):.4f}")

plt.suptitle("Top 3 Models — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_confusion_matrices.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    Accuracy : {accuracy_score(y_test, yp):.4f}")
    print(f"    F1       : {f1_score(y_test, yp, average='weighted'):.4f}")
    print(f"    Precision: {precision_score(y_test, yp, average='weighted', zero_division=0):.4f}")
    print(f"    Recall   : {recall_score(y_test, yp, average='weighted', zero_division=0):.4f}")


Detailed Metrics — Top 3:

  Logistic Regression:
    Accuracy : 0.5821
    F1       : 0.5770
    Precision: 0.5762
    Recall   : 0.5821

  FLAML:
    Accuracy : 0.5714
    F1       : 0.5693
    Precision: 0.5711
    Recall   : 0.5714

  LightGBM:
    Accuracy : 0.5707
    F1       : 0.5684
    Precision: 0.5690
    Recall   : 0.5707


## 24 · Error Analysis

Examine misclassifications from the best model.

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]

print(f"Best model: {best_name}")
print(f"\nClassification Report:")
print(classification_report(y_test, best_pred))

errors = X_test.copy()
errors["actual"] = y_test.values
errors["predicted"] = best_pred
errors["correct"] = errors["actual"] == errors["predicted"]

n_errors = (~errors["correct"]).sum()
print(f"\nTotal errors: {n_errors} / {len(errors)} ({n_errors/len(errors)*100:.1f}%)")

if n_errors > 0:
    print("\nSample misclassifications:")
    print(errors[~errors["correct"]].head(10).to_string())

Best model: Logistic Regression

Classification Report:
              precision    recall  f1-score   support

           0       0.48      0.44      0.46       490
           1       0.64      0.74      0.69       560
           2       0.60      0.53      0.56       350

    accuracy                           0.58      1400
   macro avg       0.58      0.57      0.57      1400
weighted avg       0.58      0.58      0.58      1400


Total errors: 585 / 1400 (41.8%)

Sample misclassifications:
      age_years  mileage_k  num_owners  service_records  accident_history  fuel_type  transmission  mileage_per_year  service_per_year  actual  predicted  correct
5910         18      233.8           2               24                 0        0.0           1.0         12.305263          1.263158       0          2    False
4387          6       70.1           3                9                 0        0.0           0.0         10.014286          1.285714       2          1    False
6491        

## 25 · Interpretation and Business Insight

**Key findings:**
- **Age and mileage** are the strongest condition predictors.
- **Accident history** strongly indicates poor condition.
- **Service records** are protective (well-maintained cars stay in good condition).
- **Number of owners** inversely correlates with condition.

**Business takeaway:** Prioritize low-mileage, single-owner cars with complete service histories and no accidents.

## 26 · Limitations

1. Synthetic data with simplified condition scoring.
2. No inspection-based features (rust, paint, interior).
3. Missing make/model which affects durability.
4. No price information.
5. 3-level condition is a simplification.

## 27 · How to Improve This Project

1. Use real inspection data from auction companies.
2. Add make/model and reliability ratings.
3. Include photo-based condition features (CV).
4. Add geographic factors (climate, road conditions).
5. Model condition on a continuous scale.

## 28 · Production Considerations

- Deploy as a pre-screening tool for dealerships.
- Combine with visual inspection for final assessment.
- Output condition probabilities for pricing models.
- Monitor for model drift as car technology evolves.
- Integrate with vehicle history databases.

## 29 · Common Mistakes

1. Ignoring the interaction between age and mileage.
2. Not considering make/model-specific durability.
3. Using accuracy on imbalanced conditions.
4. Treating condition as purely objective.
5. Not validating against professional inspections.

## 30 · Mini Challenge / Exercises

1. Remove `accident_history` — how does 'Poor' recall change?
2. Create `total_usage = age_years * mileage_per_year` and test.
3. Compare 3-class vs. binary (Good vs. not-Good).
4. Analyze which fuel types hold condition best.
5. Plot age vs. mileage colored by condition.

## 31 · Final Summary / Key Takeaways

1. **Age and mileage** dominate condition prediction.
2. **Accident history** strongly signals poor condition.
3. **Service records** indicate careful maintenance.
4. **Engineered ratios** (mileage/year, service/year) add value.
5. **Ordinal classification** fits the natural condition ordering.
6. **Real-world deployment** requires combining with physical inspection.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average="weighted")), 4),
        "precision": round(float(precision_score(y_test, yp, average="weighted", zero_division=0)), 4),
        "recall": round(float(recall_score(y_test, yp, average="weighted", zero_division=0)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\Classification\Used Car Condition Classification\metrics.json
{
  "CatBoost": {
    "accuracy": 0.5679,
    "f1": 0.5658,
    "precision": 0.5684,
    "recall": 0.5679
  },
  "LightGBM": {
    "accuracy": 0.5707,
    "f1": 0.5684,
    "precision": 0.569,
    "recall": 0.5707
  },
  "Logistic Regression": {
    "accuracy": 0.5821,
    "f1": 0.577,
    "precision": 0.5762,
    "recall": 0.5821
  },
  "FLAML": {
    "accuracy": 0.5714,
    "f1": 0.5693,
    "precision": 0.5711,
    "recall": 0.5714
  }
}
